In [49]:
import pandas as pd
df_raw = pd.read_csv('../data/survey.csv')
df = df_raw.copy()
df.head()

,Record ID,Site,Cohort Year,DOB,Race,Please explain,Gender,Are you a U.S. Citizen or do you have a green card and permanent resident status?,"If not a U.S. Citizen or green card holder, are you an international student on a student visa or a DACA student?",Do you consider yourself coming from an economically or environmentally disadvantaged background?,...,7,8,9,10,11,12,13,14,15,16
0,4281-1,Moscow,2019-2021,34912.0,White.,NaN,Female,Yes,NaN,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1000001,Seattle/Olympia,2019-2021,34219.0,Asian,NaN,Male,NaN,NaN,Yes,...,,,,,,,,,,
2,1000002,Seattle/Olympia,2019-2021,29809.0,Black or African American,NaN,Male,NaN,NaN,Yes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1000003,Seattle/Olympia,2019-2021,34215.0,Native Hawaiian or Other Pacific Islander. <sp...,NaN,Female,NaN,NaN,Yes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1000005,Spokane,2019-2021,34417.0,Other,NaN,Female,NaN,NaN,No,...,,,,,,,,,,


In [50]:
# I want to combine two existing colums '
#"Are you a U.S. Citizen or do you have a green card and permanent resident status?" and "If not a U.S. Citizen or green card holder, are you an international student on a student visa or a DACA student? "
#into  one column, with 3 standardized answer options 1) green card/pr 2) daca or international student 3) other
# to do this, any "yes" answers in the first exist column map to answer 1. any no's in the first followed by a yes in the second map to standardized answer 2. otherwise, 3. 
#In the end, I want this new column to exist, and neither of the earlier two. what would the code for making these changes look like?

# Original column names (shortened for readability in the code)
col_citizen = "Are you a U.S. Citizen or do you have a green card and permanent resident status?"
col_visa = "If not a U.S. Citizen or green card holder, are you an international student on a student visa or a DACA student? "

def classify_citizenship(row):
    citizen_ans = str(row[col_citizen]).strip().lower()
    visa_ans = str(row[col_visa]).strip().lower()

    if citizen_ans == 'yes':
        return 'Green card/PR'
    elif (not (citizen_ans == 'yes')  )and( visa_ans == 'yes'):
        return 'DACA or International Student'
    elif (not (citizen_ans == 'yes')) and (visa_ans == 'no'): 
        return'other'
    else:
        return 'NaN'

# Create new column
### df['citizenship_status'] = df.apply(classify_citizenship, axis=1)

# Drop the original two columns
###df = df.drop(columns=[col_citizen, col_visa])

# Verify the result
#print(df['citizenship_status'].value_counts())

#df.head()

In [51]:
# you need to end up with 3 columns - 1) rural or underserved 2) rural 3) non-rural, underserved

#combining all the columns that ask them if they have every worked in a rural/underserved area into one
col_DL = "Have volunteered in a rural or underserved community" #answers are yes no or blank. can be in different cases
col_CV = "APR.STATUS" # a set of various long options, one or two related to underserved
col_AY = "Please select the option(s) that best describe your employment or further training. Select all " \
"that apply. (choice=I am currently employed or pursuing further training in a rural setting)" # some 'checked'. very few answers, not a column worth going over everytime
col_AW = "Please select the option(s) that best describe your employment or further training. Select all that apply." \
" (choice=I am currently employed or pursuing further training in a medically underserved community)" #same as AY
col_Y = "Have you ever worked or volunteered in a rural or underserved area doing general or health-related public service?" #Yes or No. very useful answers
col_howLong = "How long did you do this work or volunteering?" #AA
col_whereAndCapacity = "Where and in what capacity? " #Z ##qualitative answers


total = 0
worked = 0

def workedRuralOrUnderserved(row):
    DL_ans = str(row[col_DL]).strip().lower()
    Y_ans = str(row[col_Y]).strip().lower()
    CV_ans = str(row[col_CV]).strip().lower() 
    AY_ans = str(row[col_AY]).strip().lower()
    AW_ans = str(row[col_AW]).strip().lower()
    
    
    if ((DL_ans == 'yes') or (Y_ans == 'yes') or (AY_ans == 'checked')or (AW_ans == 'checked')):
        
        return ('Yes')
    elif(("underserved community" in CV_ans) or ("rural setting" in CV_ans)):
        return('Yes')
    
    elif ((DL_ans == 'no') and (Y_ans == 'no')):
        return('No')
    else:
        return('NaN')


#add function for zeros to how long?

    


    

In [52]:
## did they intend to work in a rural or underserved community?
col_AH = "INTENTIONS" # relevant options = 'Rural/unincorporated area', 'Urban Underserved' for sure 
# unsure about 'Small town (population less than 2,500)', 'Town (population 2,500 to 10,000-other than suburb)'
col_AV = "Do you intend to practice in a rural and/or medically underserved setting when your residency is completed?   " # look for 'yes'. only some answwers, but not necessarily useless
col_AT = "APR.INTENTIONS" # 'Individual intends to become employed or pursue further training in a medically underserved community' or 'Individual intends to become employed or pursue further training in a rural setting'


def intentions(row):
    AH_ans = str(row[col_AH]).strip().lower()
    AV_ans = str(row[col_AV]).strip().lower()
    AT_ans = str(row[col_AT]).strip().lower()

    if (AV_ans == 'yes'):
        return('Yes')
    elif (("underserved community" in AT_ans) or ("rural setting" in AT_ans)):
        return('Yes')
    elif ((AH_ans == 'rural/incorporated area') or (AH_ans == 'Urban Underserved') or (AH_ans == 'Small town (population less than 2,500)')):
        return('Yes') #NOT SURE IF THE OPTIONS FOR THIS NEED TO BE ADJUSTED
    else:
        return('No')
        

    
    


In [53]:
# do they come from a rural or underserved community?

col_T = "Please indicate the type of community you grew up in or consider your childhood home." # same answer options as AH
col_DJ = "From a non-rural underserved community?" #check for 'yes'
col_DK = "From a rural community" #check for 'yes'

def childhoodHome(row):
    T_ans = str(row[col_T]).strip().lower()
    DJ_ans = str(row[col_DJ]).strip().lower()
    DK_ans = str(row[col_DK]).strip().lower()
    
    if ((DK_ans =='yes') or (DJ_ans == 'yes')):
        return('Yes')
    elif((T_ans == 'rural/incorporated area') or (T_ans == 'Urban Underserved') or (T_ans == 'Small town (population less than 2,500)')):
        return('Yes') #ONCE AGAIN, OPTIONS MIGHT NEED TO BE ADJUSTED
    else:
        return('No')

In [ ]:
# clean up the disadvantages stuff

#first combine some columns
col_DI = "Background - economically/environmentally disadvantaged background "
col_J = "Do you consider yourself coming from an economically or environmentally disadvantaged background?" ##!!

def combine_disadvantage1(row):
    DI_ans = str(row[col_DI]).strip().lower()
    J_ans = str(row[col_J]).strip().lower()

    if ((DI_ans == 'yes')or (J_ans == 'yes')):
        return 'Checked'

df['economically/environmentally_disadvantagedBackground'] = df.apply(combine_disadvantage1, axis=1)

col_K = "Are you the first person in your family to attend college?" #!!
col_N = "Disadvantaged status: please check as many as apply (choice=I am the first in my family to go to college)" #!!

def combine_disadvantage1(row):
    K_ans = str(row[col_K]).strip().lower()
    N_ans = str(row[col_N]).strip().lower()

    if ((K_ans == 'yes')or (N_ans == 'checked')):
        return 'Checked'
    
df['first_toAttend'] = df.apply(combine_disadvantage1, axis=1)



##now for putting them all in one column 

col_list = [
    "economically/environmentally_disadvantagedBackground",
    "first_toAttend", 
    "Disadvantaged status: please check as many as apply (choice=I grew up with English as my second language)", 
    "Disadvantaged status: please check as many as apply (choice=I have been diagnosed with a physical or mental impairment that limits my participation)", 
    "Disadvantaged status: please check as many as apply (choice=I qualify for federal/state grants which do not need to be repaid)", 
]

col_labels = {
    
}





In [55]:
# columns that I just want to drop
col_F = "Please explain" # please explain
col_CM = "APR.No IDENTIFIER" # all blank. idk what it's doing
col_CO = "APR.Record ID"
col_CP = "APR.GENDER"
col_CQ = "APR.RACE"

col_DB = "Record ID.1"
col_DE = "Site.1"
col_DF = "Cohort Year.1"
col_DG = "Gender.1"
col_DH = "Race.1"

col_CJ = "Column88"
col_CK = "Column89"

df = df.drop(columns = [col_F, col_CM, col_CO, col_CP, col_CQ, col_CJ, col_CK, col_DB, col_DE, col_DF, col_DG, col_DH])

KeyError: "['Please explain', 'APR.No IDENTIFIER', 'APR.Record ID', 'APR.GENDER', 'APR.RACE', 'Column88', 'Column89', 'Record ID.1', 'Site.1', 'Cohort Year.1', 'Gender.1', 'Race.1'] not found in axis"

In [56]:
#final 

df['citizenship_status'] = df.apply(classify_citizenship, axis=1)
df['worked_rural_orUnderserved'] = df.apply(workedRuralOrUnderserved, axis=1)
df['intended_rural_orUnderserved'] = df.apply(intentions, axis=1)
df['cameFrom_rural_orUnderserved'] = df.apply(childhoodHome, axis=1)






df = df.drop(columns=[col_citizen, col_visa])
df = df.drop(columns=[col_DL, col_Y,  col_AW, col_AY])
df = df.drop(columns = [col_AV])
df = df.drop(columns = [col_DJ, col_DK]) 
#SHOULD I DROP T and AH or is that information useful, still?


df.head()






,Record ID,Site,Cohort Year,DOB,Race,Gender,Do you consider yourself coming from an economically or environmentally disadvantaged background?,Are you the first person in your family to attend college?,"Does your father, mother, or spouse (or the previous employment, if no longer working or deceased) work in healthcare?",Who?,...,11,12,13,14,15,16,citizenship_status,worked_rural_orUnderserved,intended_rural_orUnderserved,cameFrom_rural_orUnderserved
0,4281-1,Moscow,2019-2021,34912.0,White.,Female,No,Yes,No,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Green card/PR,Yes,Yes,Yes
1,1000001,Seattle/Olympia,2019-2021,34219.0,Asian,Male,Yes,No,No,NaN,...,,,,,,,NaN,Yes,Yes,No
2,1000002,Seattle/Olympia,2019-2021,29809.0,Black or African American,Male,Yes,No,No,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,Yes,No
3,1000003,Seattle/Olympia,2019-2021,34215.0,Native Hawaiian or Other Pacific Islander. <sp...,Female,Yes,No,Yes,Mother and father,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,No,No
4,1000005,Spokane,2019-2021,34417.0,Other,Female,No,No,No,NaN,...,,,,,,,NaN,Yes,No,Yes


In [48]:
df.to_excel('cleaned_survey.xlsx', index=False)

In [ ]:

#combining identification (such as columns for race, gender, etc)